# 04 — Netherlands: Motorway AnalysisAnalyse RWS accident data. `MAXSNELHD` (speed limit) directly identifies motorway accidents — no spatial join needed.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, "../src")

from autobahn_safety.data_loaders import load_bron

DATA_RAW = Path("../data/raw")
DATA_PROC = Path("../data/processed")
DATA_PROC.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print("Ready.")

## 1. Load RWS accident data

In [ ]:
BRON_DIR = DATA_RAW / "netherlands" / "bron"

bron_years = sorted([int(f.stem.split("_")[-1]) for f in BRON_DIR.glob("bron_accidents_*.csv")])
print(f"Available BRON years: {bron_years}")

if not bron_years:
    print("No BRON data found. Run notebook 01 first.")
else:
    df_rws = load_bron(BRON_DIR, bron_years)
    df_rws["year"] = df_rws["JAAR_VKL"].astype(int)
    df_rws["MAXSNELHD_num"] = pd.to_numeric(df_rws["MAXSNELHD"], errors="coerce")
    print(f'Total: {len(df_rws):,} records, years {df_rws["year"].min()}–{df_rws["year"].max()}')
    df_rws.head(3)

## 2. Classify road types by speed limitDutch motorways (`autosnelwegen`) have a posted speed limit of 100 or 130 km/h.- 130 km/h → standard motorway (daytime)- 100 km/h → motorway with lower limit (night / environmental zone)- <100 km/h outside built-up area → N-road (`nationale weg`)- 50/30 km/h → urban

In [ ]:
def classify_road_nl(row):
    spd = row["MAXSNELHD_num"]
    bebkom = row.get("BEBKOM", "")
    if pd.isna(spd):
        return "unknown"
    spd = int(spd)
    if spd in (100, 120, 130):
        return "motorway"
    elif spd in (60, 70, 80, 90) and bebkom == "BU":
        return "n_road"
    elif spd == 50:
        return "urban_50"
    elif spd == 30:
        return "urban_30"
    else:
        return "other"


df_rws["road_type"] = df_rws.apply(classify_road_nl, axis=1)
print(df_rws["road_type"].value_counts().to_string())

In [ ]:
df_mw_nl = df_rws[df_rws["road_type"] == "motorway"].copy()
print(f"Motorway accidents: {len(df_mw_nl):,}")
print(f'Speed limits:\n{df_mw_nl["MAXSNELHD"].value_counts().to_string()}')
print(f'\nSeverity (AP3_CODE):\n{df_mw_nl["AP3_CODE"].value_counts().to_string()}')
# AP3_CODE mapping: DOD=fatal, ZGO=serious injury, LOO=slight injury, UMS=material only

## 3. Annual trends on NL motorways

In [ ]:
annual_nl = (
    df_mw_nl.groupby("year")
    .agg(
        total=("AP3_CODE", "count"),
        fatal=("AP3_CODE", lambda x: (x == "DOD").sum()),
        serious=("AP3_CODE", lambda x: (x == "ZGO").sum()),
        slight=("AP3_CODE", lambda x: (x == "LOO").sum()),
        material=("AP3_CODE", lambda x: (x == "UMS").sum()),
    )
    .reset_index()
)

annual_nl["severity_idx"] = annual_nl["fatal"] / annual_nl["total"]
annual_nl

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(
    annual_nl["year"], annual_nl["total"], marker="o", color="#2a9d8f", linewidth=2, label="Total"
)
axes[0].plot(
    annual_nl["year"],
    annual_nl["serious"],
    marker="s",
    color="#e9c46a",
    linewidth=2,
    label="Serious injury",
)
axes[0].plot(
    annual_nl["year"], annual_nl["fatal"], marker="^", color="crimson", linewidth=2, label="Fatal"
)
axes[0].set_title("NL motorway accidents by severity")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Accidents")
axes[0].legend()

axes[1].plot(
    annual_nl["year"], annual_nl["severity_idx"] * 1000, marker="o", color="crimson", linewidth=2
)
axes[1].set_title("NL motorway: severity index (fatalities per 1000 accidents)")
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Fatalities per 1000 accidents")

plt.suptitle("Netherlands motorway safety (BRON)", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Save processed NL data

In [ ]:
out = DATA_PROC / "rws_motorway_annual.parquet"
annual_nl.to_parquet(out, index=False)
print(f"Saved to {out}")
annual_nl